In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import requests
import yfinance as yf

print("Python:", sys.version.split()[0])
print("Interpreter:", sys.executable)
print("Working directory:", Path.cwd())
print()
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Requests:", requests.__version__)
print("yfinance:", yf.__version__)

Python: 3.14.6
Interpreter: /Users/jonathanma/Desktop/Projects/lev-v-prof/.venv/bin/python
Working directory: /Users/jonathanma/Desktop/Projects/lev-v-prof/code

NumPy: 2.5.5
Pandas: 3.1.1
Requests: 2.34.2
yfinance: 1.8.0


## 02 — SEC API Connectivity
This section verifies access to the SEC EDGAR API and examines
the company identifier data used to construct our manufacturing
firm universe.

In [ ]:
import requests
import pandas as pd

SEC_HEADERS = {"User-Agent": "Leverage Profitability Research rucstudentfinanceassociation@gmail.com", "Accept-Encoding": "gzip, deflate",}
SEC_TICKERS_URL = ("https://www.sec.gov/files/company_tickers.json")
response = requests.get(SEC_TICKERS_URL, headers=SEC_HEADERS,timeout=30)
print("HTTP status:", response.status_code)
response.raise_for_status()

companies = pd.DataFrame(response.json().values())
companies["cik_str"] = (companies["cik_str"].astype(str).str.zfill(10))

print("Companies retrieved:", len(companies))
display(companies.head(10))

HTTP status: 200
Companies retrieved: 10398


,cik_str,ticker,title
0,0000320193,AAPL,Apple Inc.
1,0000789019,MSFT,MICROSOFT CORP
2,0001045810,NVDA,NVIDIA CORP
3,0001652044,GOOGL,Alphabet Inc.
4,0001018724,AMZN,AMAZON COM INC
5,0001730168,AVGO,Broadcom Inc.
6,0001326801,META,"Meta Platforms, Inc."
7,0001318605,TSLA,"Tesla, Inc."
8,0001067983,BRK-B,BERKSHIRE HATHAWAY INC
9,0000059478,LLY,ELI LILLY & Co


## 03 — Manufacturing Firm Identification

We identify publicly traded manufacturing firms using Standard
Industrial Classification (SIC) codes 2000–3999.

The SEC ticker dataset provides company identifiers but does not
include SIC classifications. We therefore retrieve company metadata
from the SEC submissions API.

This section first tests the procedure on a small sample before
scaling to the broader company universe.

In [ ]:
import time

TEST_TICKERS = ["AAPL", "MSFT", "TSLA", "CAT", "LLY"]

def get_company_metadata(cik):
    """Retrieve company metadata from the SEC submissions API."""
    url = ("https://data.sec.gov/submissions/"f"CIK{str(cik).zfill(10)}.json")
    response = requests.get(url, headers=SEC_HEADERS, timeout=30)
    response.raise_for_status()
    data = response.json()
    return {
        "cik_str": str(cik).zfill(10),
        "name": data.get("name"),
        "sic": data.get("sic"),
        "sic_description": data.get("sicDescription"),
        "state_of_incorporation": data.get("stateOfIncorporation"),
    }

results = []

for ticker in TEST_TICKERS:
    match = companies.loc[companies["ticker"] == ticker]
    if match.empty:
        print(f"Ticker not found: {ticker}")
        continue
    cik = match.iloc[0]["cik_str"]
    try:
        metadata = get_company_metadata(cik)
        metadata["ticker"] = ticker
        results.append(metadata)
    except requests.RequestException as exc:
        print(f"Failed to retrieve {ticker}: {exc}")
    time.sleep(0.2)


sic_test = pd.DataFrame(results)

sic_test["is_manufacturing"] = (
    pd.to_numeric(sic_test["sic"], errors="coerce")
    .between(2000, 3999)
)

display(sic_test)

,cik_str,name,sic,sic_description,state_of_incorporation,ticker,is_manufacturing
0,0000320193,Apple Inc.,3571,Electronic Computers,CA,AAPL,True
1,0000789019,MICROSOFT CORP,7372,Services-Prepackaged Software,WA,MSFT,False
2,0001318605,"Tesla, Inc.",3711,Motor Vehicles & Passenger Car Bodies,TX,TSLA,True
3,0000018230,CATERPILLAR INC,3531,Construction Machinery & Equip,DE,CAT,True
4,0000059478,ELI LILLY & Co,2834,Pharmaceutical Preparations,IN,LLY,True


## 04 — Company Universe and Exchange Metadata

This section examines the SEC's bulk company identifier and
exchange information to support construction of the manufacturing
firm universe.

The objective is to identify an efficient, reproducible method
for obtaining SIC classifications without unnecessarily requesting
thousands of individual company submissions.

In [ ]:
SEC_EXCHANGE_URL = ("https://www.sec.gov/files/company_tickers_exchange.json")

response = requests.get(SEC_EXCHANGE_URL, headers=SEC_HEADERS, timeout=30)
response.raise_for_status()
exchange_data = response.json()
exchange_companies = pd.DataFrame(exchange_data["data"], columns=exchange_data["fields"])

display(exchange_companies.head(10))
print("Records:", len(exchange_companies))
print("Columns:", exchange_companies.columns.tolist())
print("Unique CIKs:", exchange_companies["cik"].nunique())

,cik,name,ticker,exchange
0,320193,Apple Inc.,AAPL,Nasdaq
1,789019,MICROSOFT CORP,MSFT,Nasdaq
2,1652044,Alphabet Inc.,GOOGL,Nasdaq
3,1045810,NVIDIA CORP,NVDA,Nasdaq
4,1018724,AMAZON COM INC,AMZN,Nasdaq
5,1730168,Broadcom Inc.,AVGO,Nasdaq
6,1326801,"Meta Platforms, Inc.",META,Nasdaq
7,1318605,"Tesla, Inc.",TSLA,Nasdaq
8,1067983,BERKSHIRE HATHAWAY INC,BRK-B,NYSE
9,59478,ELI LILLY & Co,LLY,NYSE


Records: 10400
Columns: ['cik', 'name', 'ticker', 'exchange']
Unique CIKs: 7988


## 05 — Unique Company Universe

The SEC exchange dataset contains multiple ticker records for some
companies. Since the unit of analysis is the firm, we construct
a unique company universe using CIK identifiers.

Ticker and exchange information are retained as metadata, while
financial statements and SIC classifications will be retrieved
at the company level.

In [ ]:
# Standardize identifiers
exchange_companies["cik_str"] = (exchange_companies["cik"].astype(str).str.zfill(10))

# Count exchange listings
print("Exchange distribution:")
display(
    exchange_companies["exchange"]
    .value_counts(dropna=False)
    .rename_axis("exchange")
    .reset_index(name="n_ticker_records")
)

# Aggregate tickers and exchanges for each company
company_universe = (
    exchange_companies
    .groupby("cik_str", as_index=False)
    .agg(
        name=("name", "first"),
        tickers=("ticker", lambda x: sorted(set(x.dropna()))),
        exchanges=("exchange", lambda x: sorted(set(x.dropna())))
    )
)

print("Unique companies:", len(company_universe))
print("Duplicate CIKs:", company_universe["cik_str"].duplicated().sum())

display(company_universe.head(10))

Exchange distribution:


,exchange,n_ticker_records
0,Nasdaq,4353
1,NYSE,3258
2,OTC,2574
3,NaN,157
4,CBOE,44
5,TXSE,14


Unique companies: 7988
Duplicate CIKs: 0


,cik_str,name,tickers,exchanges
0,0000001750,AAR CORP,[AIR],[NYSE]
1,0000001800,ABBOTT LABORATORIES,[ABT],[NYSE]
2,0000002098,ACME UNITED CORP,[ACU],[NYSE]
3,0000002186,BK Technologies Corp,[BKTI],[NYSE]
4,0000002230,"ADAMS DIVERSIFIED EQUITY FUND, INC.",[ADX],[NYSE]
5,0000002488,ADVANCED MICRO DEVICES INC,[AMD],[Nasdaq]
6,0000002809,AGNICO EAGLE MINES LTD,"[AEM, AEMRF]","[NYSE, OTC]"
7,0000002969,"Air Products & Chemicals, Inc.",[APD],[NYSE]
8,0000003124,AKZO NOBEL NV,"[AKZOF, AKZOY]",[OTC]
9,0000003197,CECO ENVIRONMENTAL CORP,[CECO],[TXSE]


## 06 — Batch SIC Classification

We retrieve SEC industry classifications for a pilot sample of
100 unique companies.

Manufacturing firms are provisionally defined by SIC codes
2000–3999. This initial batch evaluates API reliability,
classification coverage, and the proportion of firms belonging
to the manufacturing sector.

The pilot does not yet constitute the final research sample.

In [ ]:
import time
from tqdm.auto import tqdm

PILOT_SIZE = 100
# Use a reproducible random sample rather than the first 100 CIKs
pilot_companies = company_universe.sample(n=PILOT_SIZE,random_state=42).copy()

sic_results = []
sic_errors = []

for row in tqdm(
    pilot_companies.itertuples(index=False),
    total=len(pilot_companies),
    desc="Retrieving SEC metadata"
):
    try:
        metadata = get_company_metadata(row.cik_str)
        metadata["tickers"] = row.tickers
        metadata["exchanges"] = row.exchanges
        sic_results.append(metadata)
    except requests.RequestException as exc:
        sic_errors.append({"cik_str": row.cik_str,"error": str(exc)})
    # Conservative request pacing
    time.sleep(0.2)

sic_pilot = pd.DataFrame(sic_results)

if not sic_pilot.empty:
    sic_pilot["sic"] = pd.to_numeric(sic_pilot["sic"],errors="coerce")
    sic_pilot["is_manufacturing"] = (sic_pilot["sic"].between(2000, 3999))

print(f"Requested: {len(pilot_companies)}")
print(f"Successful: {len(sic_results)}")
print(f"Failed: {len(sic_errors)}")

if not sic_pilot.empty:
    print(f"Missing SIC: {sic_pilot['sic'].isna().sum()}")
    print("Manufacturing firms:",sic_pilot["is_manufacturing"].sum())
    display(sic_pilot[["cik_str","name","sic","sic_description","is_manufacturing"]].head(15))

/Users/jonathanma/Desktop/Projects/lev-v-prof/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Retrieving SEC metadata: 100%|██████████| 100/100 [00:56<00:00,  1.77it/s]

Requested: 100
Successful: 100
Failed: 0
Missing SIC: 17
Manufacturing firms: 26


,cik_str,name,sic,sic_description,is_manufacturing
0,0001337298,FutureFuel Corp.,2860.0,Industrial Organic Chemicals,True
1,0001818844,"Dogwood Therapeutics, Inc.",2834.0,Pharmaceutical Preparations,True
2,0001290658,"IIOT-OXYS, Inc.",7372.0,Services-Prepackaged Software,False
3,0001082324,VirnetX Holding Corp,6794.0,Patent Owners & Lessors,False
4,0001698506,Simwon America Corp.,3711.0,Motor Vehicles & Passenger Car Bodies,True
5,0001013880,"TTEC Holdings, Inc.",7363.0,Services-Help Supply Services,False
6,0002145089,WINDWARD BIO GROUP AG,2836.0,"Biological Products, (No Diagnostic Substances)",True
7,0000845982,SMITH & NEPHEW PLC,3842.0,"Orthopedic, Prosthetic & Surgical Appliances &...",True
8,0002034400,Black Titan Corp,7371.0,Services-Computer Programming Services,False
9,0001477960,LataMed AI Corp.,7370.0,"Services-Computer Programming, Data Processing...",False


## 07 — Full SIC Classification

This section retrieves SIC classifications for the complete
unique-company universe.

The collection process uses incremental checkpoints, request
pacing, and retry handling to support reproducibility and
recovery from interrupted sessions.

Results are stored in `data/raw/`, preserving the original
SEC metadata alongside company identifiers.

In [7]:
import json
import time
from pathlib import Path
import pandas as pd
import requests
from tqdm.auto import tqdm

RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT = RAW_DIR / "sec_company_metadata.jsonl"

BATCH_SIZE = 100
REQUEST_DELAY = 0.25
MAX_RETRIES = 3

session = requests.Session()
session.headers.update(SEC_HEADERS)

def fetch_sec_metadata(cik):
    """Retrieve SEC company metadata with retry handling."""
    cik = str(cik).zfill(10)
    url = ("https://data.sec.gov/submissions/"f"CIK{cik}.json")
    for attempt in range(MAX_RETRIES):
        try:
            response = session.get(url, timeout=30)
            if response.status_code in (429, 500, 502, 503, 504):
                response.raise_for_status()
            response.raise_for_status()
            data = response.json()
            return {
                "cik_str": cik,
                "name": data.get("name"),
                "sic": data.get("sic"),
                "sic_description": data.get("sicDescription"),
                "state_of_incorporation": data.get(
                    "stateOfIncorporation"
                ),
                "state_of_business": (
                    data.get("addresses", {})
                    .get("business", {})
                    .get("stateOrCountry")
                ),
            }
        except (requests.RequestException, ValueError):
            if attempt == MAX_RETRIES - 1:
                raise
            time.sleep(2 ** (attempt + 1))

# Load existing checkpoint, if available
completed = {}
if CHECKPOINT.exists():
    with CHECKPOINT.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                record = json.loads(line)
                completed[record["cik_str"]] = record

print("Previously completed:", len(completed))

# Deduplicated SEC universe
pending = company_universe.loc[~company_universe["cik_str"].isin(completed)].copy()
print("Remaining companies:", len(pending))

errors = []

# Append successful records immediately
with CHECKPOINT.open("a", encoding="utf-8") as f:
    for start in range(0, len(pending), BATCH_SIZE):
        batch = pending.iloc[start:start + BATCH_SIZE]
        for row in tqdm(
            batch.itertuples(index=False),
            total=len(batch),
            desc=f"Batch {start // BATCH_SIZE + 1}"
        ):
            try:
                record = fetch_sec_metadata(row.cik_str)
                f.write(json.dumps(record) + "\n")
                f.flush()
                completed[record["cik_str"]] = record
            except (requests.RequestException, ValueError) as exc:
                errors.append({
                    "cik_str": row.cik_str,
                    "error": str(exc)
                })
            time.sleep(REQUEST_DELAY)
        print(
            f"Completed: {len(completed):,} / "
            f"{len(company_universe):,}"
        )

# Build the classification dataset
sec_metadata = pd.DataFrame(completed.values())
sec_metadata["sic"] = pd.to_numeric(sec_metadata["sic"],errors="coerce")
sec_metadata["is_manufacturing"] = (sec_metadata["sic"].between(2000, 3999))
manufacturing_firms = sec_metadata.loc[sec_metadata["is_manufacturing"]].copy()

print("\nClassification summary")
print("----------------------")
print("Companies classified:", len(sec_metadata))
print("Missing SIC:", sec_metadata["sic"].isna().sum())
print("Manufacturing firms:", len(manufacturing_firms))
print("Failed requests:", len(errors))

display(manufacturing_firms.head(10))


Previously completed: 0
Remaining companies: 7988


Batch 1: 100%|██████████| 100/100 [00:38<00:00,  2.61it/s]


Completed: 100 / 7,988


Batch 2: 100%|██████████| 100/100 [00:37<00:00,  2.63it/s]


Completed: 200 / 7,988


Batch 3: 100%|██████████| 100/100 [00:37<00:00,  2.68it/s]


... (152 outputs left out)

Completed: 7,900 / 7,988


Batch 80: 100%|██████████| 88/88 [00:36<00:00,  2.41it/s]

Completed: 7,988 / 7,988

Classification summary
----------------------
Companies classified: 7988
Missing SIC: 1102
Manufacturing firms: 2407
Failed requests: 0


,cik_str,name,sic,sic_description,state_of_incorporation,state_of_business,is_manufacturing
0,0000001750,AAR CORP,3720.0,Aircraft & Parts,DE,IL,True
1,0000001800,ABBOTT LABORATORIES,2834.0,Pharmaceutical Preparations,IL,IL,True
2,0000002098,ACME UNITED CORP,3420.0,"Cutlery, Handtools & General Hardware",CT,CT,True
3,0000002186,BK Technologies Corp,3663.0,Radio & Tv Broadcasting & Communications Equip...,NV,FL,True
5,0000002488,ADVANCED MICRO DEVICES INC,3674.0,Semiconductors & Related Devices,DE,CA,True
7,0000002969,"Air Products & Chemicals, Inc.",2810.0,Industrial Inorganic Chemicals,DE,PA,True
8,0000003124,AKZO NOBEL NV,2800.0,Chemicals & Allied Products,,NaN,True
9,0000003197,CECO ENVIRONMENTAL CORP,3564.0,Industrial & Commercial Fans & Blowers & Air P...,DE,TX,True
14,0000004127,"SKYWORKS SOLUTIONS, INC.",3674.0,Semiconductors & Related Devices,DE,CA,True
15,0000004281,Howmet Aerospace Inc.,3350.0,Rolling Drawing & Extruding of Nonferrous Metals,DE,PA,True


## 08 — Financial Statement Feasibility

This section evaluates the availability of financial statement
variables required to study the leverage–profitability relationship.

We retrieve SEC XBRL company facts for a pilot sample of 30
SIC-classified manufacturing firms.

The initial objective is to assess whether key accounting concepts
are reported using standardized US-GAAP tags. Fiscal-year alignment,
accounting definitions, and ratio construction will be evaluated
in subsequent steps.

In [10]:

# 08 — SEC XBRL financial statement feasibility

import time
import pandas as pd
import requests
from tqdm.auto import tqdm

FINANCIAL_PILOT_SIZE = 30

financial_pilot = manufacturing_firms.sample(
    n=FINANCIAL_PILOT_SIZE,
    random_state=42
).copy()

# Candidate XBRL tags.
# These are not yet final accounting definitions.
FINANCIAL_TAGS = {
    "assets": [
        "Assets"
    ],
    "net_income": [
        "NetIncomeLoss"
    ],
    "equity": [
        "StockholdersEquity",
        "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest"
    ],
    "revenue": [
        "RevenueFromContractWithCustomerExcludingAssessedTax",
        "Revenues",
        "SalesRevenueNet"
    ],
    "long_term_debt": [
        "LongTermDebtNoncurrent",
        "LongTermDebt"
    ],
    "current_debt": [
        "LongTermDebtCurrent",
        "ShortTermBorrowings"
    ],
    "total_liabilities": [
        "Liabilities"
    ]
}


def get_company_facts(cik):
    """Retrieve standardized SEC XBRL company facts."""

    url = (
        "https://data.sec.gov/api/xbrl/companyfacts/"
        f"CIK{str(cik).zfill(10)}.json"
    )

    response = requests.get(
        url,
        headers=SEC_HEADERS,
        timeout=30
    )
    response.raise_for_status()

    return response.json()


coverage_results = []
coverage_errors = []

for row in tqdm(
    financial_pilot.itertuples(index=False),
    total=len(financial_pilot),
    desc="Checking financial coverage"
):
    try:
        facts = get_company_facts(row.cik_str)
        gaap = facts.get("facts", {}).get("us-gaap", {})

        record = {
            "cik_str": row.cik_str,
            "name": row.name,
            "sic": row.sic
        }

        for variable, tags in FINANCIAL_TAGS.items():
            record[variable] = any(
                tag in gaap and
                bool(gaap[tag].get("units", {}).get("USD", []))
                for tag in tags
            )

        coverage_results.append(record)

    except requests.RequestException as exc:
        coverage_errors.append({
            "cik_str": row.cik_str,
            "error": str(exc)
        })

    time.sleep(0.2)


financial_coverage = pd.DataFrame(coverage_results)

coverage_summary = (
    financial_coverage[list(FINANCIAL_TAGS.keys())]
    .mean()
    .mul(100)
    .round(1)
    .rename_axis("variable")
    .reset_index(name="coverage_pct")
)

print("Requested:", len(financial_pilot))
print("Successful:", len(financial_coverage))
print("Failed:", len(coverage_errors))

display(coverage_summary)


Checking financial coverage: 100%|██████████| 30/30 [00:16<00:00,  1.87it/s]

Requested: 30
Successful: 30
Failed: 0


,variable,coverage_pct
0,assets,93.3
1,net_income,90.0
2,equity,93.3
3,revenue,90.0
4,long_term_debt,56.7
5,current_debt,50.0
6,total_liabilities,90.0


## 09 — Investigating Debt-Related XBRL Tags

The initial financial feasibility analysis found substantially lower
coverage for current and long-term debt than for other accounting
variables.

This section examines the debt-related US-GAAP tags reported by
sampled manufacturing firms to identify alternative accounting
concepts and evaluate whether the original tag mapping is too
restrictive.

Tag discovery is exploratory. Tags with similar names are not
necessarily interchangeable, and no debt values are combined at
this stage.

In [11]:
# 09 — Discover debt-related US-GAAP tags

from collections import Counter

DEBT_KEYWORDS = (
    "Debt",
    "Borrowing",
    "Loan",
    "CreditFacility",
    "CommercialPaper",
    "NotesPayable"
)

debt_tag_counts = Counter()
debt_tag_examples = {}

errors = []

for row in tqdm(
    financial_pilot.itertuples(index=False),
    total=len(financial_pilot),
    desc="Inspecting debt tags"
):
    try:
        facts = get_company_facts(row.cik_str)
        gaap = facts.get("facts", {}).get("us-gaap", {})

        for tag, details in gaap.items():
            if not any(word in tag for word in DEBT_KEYWORDS):
                continue

            # Retain monetary concepts only
            if "USD" not in details.get("units", {}):
                continue

            debt_tag_counts[tag] += 1

            debt_tag_examples.setdefault(
                tag, []
            ).append(row.name)

    except requests.RequestException as exc:
        errors.append({
            "cik_str": row.cik_str,
            "error": str(exc)
        })

    time.sleep(0.2)


debt_tag_summary = pd.DataFrame(
    [
        {
            "tag": tag,
            "firms_reporting": count,
            "coverage_pct": round(
                100 * count / len(financial_pilot), 1
            ),
            "example_firms": ", ".join(
                debt_tag_examples[tag][:3]
            )
        }
        for tag, count in debt_tag_counts.items()
    ]
).sort_values(
    "firms_reporting",
    ascending=False
)

print("Unique debt-related tags:", len(debt_tag_summary))
print("Failed requests:", len(errors))

display(debt_tag_summary.head(30))

Inspecting debt tags: 100%|██████████| 30/30 [00:22<00:00,  1.36it/s]

Unique debt-related tags: 200
Failed requests: 0


,tag,firms_reporting,coverage_pct,example_firms
14,LongTermDebt,14,46.7,"J.Jill, Inc., Leatt Corp, AtriCure, Inc."
3,GainsLossesOnExtinguishmentOfDebt,14,46.7,"Unusual Machines, Inc., J.Jill, Inc., Caribou ..."
15,LongTermDebtCurrent,13,43.3,"J.Jill, Inc., Compass Therapeutics, Inc., Atri..."
58,LongTermDebtMaturitiesRepaymentsOfPrincipalInY...,12,40.0,"Leatt Corp, AtriCure, Inc., TYSON FOODS, INC."
57,LongTermDebtMaturitiesRepaymentsOfPrincipalInY...,12,40.0,"Leatt Corp, AtriCure, Inc., TYSON FOODS, INC."
56,LongTermDebtMaturitiesRepaymentsOfPrincipalInN...,12,40.0,"Leatt Corp, AtriCure, Inc., TYSON FOODS, INC."
18,ProceedsFromIssuanceOfLongTermDebt,12,40.0,"J.Jill, Inc., Leatt Corp, AtriCure, Inc."
17,PaymentsOfDebtIssuanceCosts,12,40.0,"J.Jill, Inc., Cricut, Inc., AtriCure, Inc."
16,LongTermDebtNoncurrent,12,40.0,"J.Jill, Inc., Compass Therapeutics, Inc., Atri..."
89,LongTermDebtMaturitiesRepaymentsOfPrincipalInY...,11,36.7,"AtriCure, Inc., TYSON FOODS, INC., e.l.f. Beau..."


## 10 — Inspecting Fiscal-Year Debt Observations

This section examines the actual XBRL observations associated
with debt-related balance-sheet concepts.

We focus on fiscal year 2024 and inspect reporting dates,
filing forms, units, and values before defining a general
procedure for constructing total debt.

The objective is to distinguish accounting concepts that
represent consolidated debt balances from overlapping
components or instrument-specific disclosures.

In [12]:

# 10 — Inspect fiscal-year debt observations

import pandas as pd

INSPECTION_TICKERS = ["AAPL", "CAT", "LLY"]

DEBT_INSPECTION_TAGS = [
    "LongTermDebt",
    "LongTermDebtCurrent",
    "LongTermDebtNoncurrent",
    "ShortTermBorrowings",
    "ShortTermDebt",
    "CommercialPaper",
    "LongTermDebtAndCapitalLeaseObligations",
    "LongTermDebtAndCapitalLeaseObligationsCurrent",
    "LongTermDebtAndCapitalLeaseObligationsNoncurrent",
    "DebtCurrent",
    "DebtNoncurrent",
    "DebtInstrumentCarryingAmount",
]


def inspect_debt_facts(ticker, fiscal_year=2024):
    """Inspect debt facts reported in 10-K filings."""

    match = companies.loc[
        companies["ticker"] == ticker
    ]

    if match.empty:
        raise ValueError(f"Ticker not found: {ticker}")

    cik = match.iloc[0]["cik_str"]
    facts = get_company_facts(cik)

    gaap = facts.get("facts", {}).get("us-gaap", {})
    rows = []

    for tag in DEBT_INSPECTION_TAGS:
        if tag not in gaap:
            continue

        observations = (
            gaap[tag]
            .get("units", {})
            .get("USD", [])
        )

        for obs in observations:
            if (
                obs.get("form") == "10-K"
                and obs.get("fy") == fiscal_year
            ):
                rows.append({
                    "ticker": ticker,
                    "tag": tag,
                    "value": obs.get("val"),
                    "start": obs.get("start"),
                    "end": obs.get("end"),
                    "fy": obs.get("fy"),
                    "fp": obs.get("fp"),
                    "frame": obs.get("frame"),
                    "filed": obs.get("filed"),
                    "accession": obs.get("accn"),
                })

    return pd.DataFrame(rows)


inspection_results = []

for ticker in INSPECTION_TICKERS:
    df = inspect_debt_facts(ticker)

    if not df.empty:
        inspection_results.append(df)

if inspection_results:
    debt_inspection = pd.concat(
        inspection_results,
        ignore_index=True
    )

    debt_inspection = debt_inspection.sort_values(
        ["ticker", "tag", "end", "filed"]
    )

    display(debt_inspection)
else:
    print("No matching observations found.")


,ticker,tag,value,start,end,fy,fp,frame,filed,accession
6,AAPL,CommercialPaper,5985000000,None,2023-09-30,2024,FY,CY2023Q3I,2024-11-01,0000320193-24-000123
7,AAPL,CommercialPaper,9967000000,None,2024-09-28,2024,FY,NaN,2024-11-01,0000320193-24-000123
8,AAPL,DebtInstrumentCarryingAmount,106572000000,None,2023-09-30,2024,FY,CY2023Q3I,2024-11-01,0000320193-24-000123
9,AAPL,DebtInstrumentCarryingAmount,97341000000,None,2024-09-28,2024,FY,NaN,2024-11-01,0000320193-24-000123
0,AAPL,LongTermDebt,105103000000,None,2023-09-30,2024,FY,CY2023Q3I,2024-11-01,0000320193-24-000123
1,AAPL,LongTermDebt,96662000000,None,2024-09-28,2024,FY,NaN,2024-11-01,0000320193-24-000123
2,AAPL,LongTermDebtCurrent,9822000000,None,2023-09-30,2024,FY,CY2023Q3I,2024-11-01,0000320193-24-000123
3,AAPL,LongTermDebtCurrent,10912000000,None,2024-09-28,2024,FY,NaN,2024-11-01,0000320193-24-000123
4,AAPL,LongTermDebtNoncurrent,95281000000,None,2023-09-30,2024,FY,CY2023Q3I,2024-11-01,0000320193-24-000123
5,AAPL,LongTermDebtNoncurrent,85750000000,None,2024-09-28,2024,FY,NaN,2024-11-01,0000320193-24-000123


## 11 — Debt Component Consistency

This section evaluates whether reported long-term debt balances
are consistent with their current and noncurrent components.

We restrict attention to balance-sheet observations from fiscal
year 2024 and compare values reported for the same period.

The objective is to identify potential double-counting and
establish a defensible debt-variable construction strategy.

In [13]:

# 11 — Long-term debt accounting identity diagnostic

import numpy as np
import pandas as pd

IDENTITY_TAGS = [
    "LongTermDebt",
    "LongTermDebtCurrent",
    "LongTermDebtNoncurrent",
]


def extract_2024_debt_components(cik):
    """Extract debt components from a fiscal-2024 10-K."""

    facts = get_company_facts(cik)
    gaap = facts.get("facts", {}).get("us-gaap", {})

    # Find the latest original fiscal-2024 10-K accession.
    filings = []

    for tag in IDENTITY_TAGS:
        if tag not in gaap:
            continue

        for obs in gaap[tag].get("units", {}).get("USD", []):
            if (
                obs.get("form") == "10-K"
                and obs.get("fy") == 2024
                and obs.get("fp") == "FY"
            ):
                filings.append({
                    "accn": obs.get("accn"),
                    "filed": obs.get("filed"),
                })

    if not filings:
        return None

    filing = max(filings, key=lambda x: x["filed"])
    accession = filing["accn"]

    rows = []

    for tag in IDENTITY_TAGS:
        if tag not in gaap:
            continue

        for obs in gaap[tag].get("units", {}).get("USD", []):
            if (
                obs.get("accn") == accession
                and obs.get("form") == "10-K"
                and obs.get("fy") == 2024
                and obs.get("fp") == "FY"
                and obs.get("start") is None
                and obs.get("end", "").startswith("2024")
            ):
                rows.append({
                    "tag": tag,
                    "value": obs["val"],
                    "end": obs["end"],
                })

    if not rows:
        return None

    df = pd.DataFrame(rows)

    # Use the latest balance-sheet date in 2024.
    fiscal_end = df["end"].max()
    df = df.loc[df["end"] == fiscal_end]

    values = df.drop_duplicates("tag").set_index("tag")["value"]

    return {
        "fiscal_end": fiscal_end,
        "long_term_debt": values.get("LongTermDebt", np.nan),
        "current_portion": values.get("LongTermDebtCurrent", np.nan),
        "noncurrent_portion": values.get(
            "LongTermDebtNoncurrent", np.nan
        ),
    }


identity_results = []

for row in tqdm(
    financial_pilot.itertuples(index=False),
    total=len(financial_pilot),
    desc="Checking debt identities"
):
    try:
        result = extract_2024_debt_components(row.cik_str)

        if result is not None:
            result["cik_str"] = row.cik_str
            result["name"] = row.name
            identity_results.append(result)

    except requests.RequestException as exc:
        print(f"Failed: {row.cik_str}: {exc}")

    time.sleep(0.2)


debt_identity = pd.DataFrame(identity_results)

if not debt_identity.empty:
    debt_identity["component_sum"] = (
        debt_identity["current_portion"]
        + debt_identity["noncurrent_portion"]
    )

    debt_identity["difference"] = (
        debt_identity["long_term_debt"]
        - debt_identity["component_sum"]
    )

    debt_identity["relative_difference"] = (
        debt_identity["difference"].abs()
        / debt_identity["long_term_debt"].abs().replace(0, np.nan)
    )

    complete = debt_identity.dropna(
        subset=[
            "long_term_debt",
            "current_portion",
            "noncurrent_portion"
        ]
    )

    print("Firms with debt observations:", len(debt_identity))
    print("Firms with all three components:", len(complete))

    if not complete.empty:
        print(
            "Approximately consistent identities:",
            (complete["relative_difference"] < 0.01).sum()
        )

        display(
            complete[
                [
                    "name",
                    "long_term_debt",
                    "component_sum",
                    "difference",
                    "relative_difference"
                ]
            ]
        )


Checking debt identities: 100%|██████████| 30/30 [00:19<00:00,  1.52it/s]

Firms with debt observations: 13
Firms with all three components: 4
Approximately consistent identities: 4


,name,long_term_debt,component_sum,difference,relative_difference
5,V F CORP,5.686846e+09,5.686846e+09,0.0,0.0
7,nVent Electric plc,2.155000e+09,2.155000e+09,0.0,0.0
9,"ARVINAS, INC.",8.000000e+05,8.000000e+05,0.0,0.0
10,WESTINGHOUSE AIR BRAKE TECHNOLOGIES CORP,3.980000e+09,3.980000e+09,0.0,0.0


## 12 — Fiscal-Year 2024 Financial Data Availability

This section evaluates whether the sampled manufacturing firms
report the accounting variables needed for the proposed
cross-sectional analysis.

Unlike the initial historical tag-coverage analysis, this
diagnostic restricts attention to observations corresponding
to fiscal-year 2024.

The analysis distinguishes balance-sheet variables, measured
at fiscal year-end, from income-statement variables, measured
over the fiscal year.

Debt coverage is evaluated using a broader set of candidate
concepts. Their availability does not yet establish that
total interest-bearing debt can be constructed consistently.

In [14]:
import pandas as pd
import time
from tqdm.auto import tqdm

BALANCE_TAGS = {
    "assets": ["Assets"],
    "equity": [
        "StockholdersEquity",
        "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest"
    ],
    "debt_aggregate": [
        "LongTermDebt",
        "DebtCurrent",
        "DebtNoncurrent"
    ],
    "debt_components": [
        "LongTermDebtCurrent",
        "LongTermDebtNoncurrent",
        "ShortTermBorrowings",
        "ShortTermDebt",
        "CommercialPaper",
        "LongTermDebtAndCapitalLeaseObligations",
        "LongTermDebtAndCapitalLeaseObligationsCurrent",
        "LongTermDebtAndCapitalLeaseObligationsNoncurrent"
    ]
}

INCOME_TAGS = {
    "net_income": [
        "NetIncomeLoss",
        "ProfitLoss"
    ],
    "revenue": [
        "RevenueFromContractWithCustomerExcludingAssessedTax",
        "Revenues",
        "SalesRevenueNet"
    ]
}


def check_fy2024_coverage(cik):
    """Check fiscal-2024 observations in SEC Company Facts."""

    facts = get_company_facts(cik)
    gaap = facts.get("facts", {}).get("us-gaap", {})

    # Collect fiscal-year-end dates from FY2024 10-K
    # balance-sheet asset observations.
    asset_records = []

    if "Assets" in gaap:
        asset_records = [
            obs
            for obs in gaap["Assets"].get("units", {}).get("USD", [])
            if (
                obs.get("form") == "10-K"
                and obs.get("fy") == 2024
                and obs.get("fp") == "FY"
                and obs.get("start") is None
                and obs.get("end", "").startswith("2024")
            )
        ]

    if not asset_records:
        return None

    # Choose the latest FY2024 fiscal-year-end observation.
    target = max(
        asset_records,
        key=lambda x: (x["end"], x.get("filed", ""))
    )

    fiscal_end = target["end"]
    accession = target["accn"]

    result = {
        "fiscal_end": fiscal_end,
        "accession": accession
    }

    # Balance-sheet concepts must match the fiscal-year-end
    # and the selected filing.
    for variable, tags in BALANCE_TAGS.items():

        result[variable] = any(
            any(
                obs.get("accn") == accession
                and obs.get("end") == fiscal_end
                and obs.get("start") is None
                for obs in gaap.get(tag, {})
                    .get("units", {})
                    .get("USD", [])
            )
            for tag in tags
        )

    # Income-statement concepts must span approximately
    # one fiscal year and end on the same date.
    for variable, tags in INCOME_TAGS.items():

        found = False

        for tag in tags:
            observations = (
                gaap.get(tag, {})
                .get("units", {})
                .get("USD", [])
            )

            for obs in observations:
                if (
                    obs.get("accn") == accession
                    and obs.get("end") == fiscal_end
                    and obs.get("start") is not None
                ):
                    days = (
                        pd.Timestamp(obs["end"])
                        - pd.Timestamp(obs["start"])
                    ).days

                    if 330 <= days <= 380:
                        found = True
                        break

            if found:
                break

        result[variable] = found

    result["has_profitability"] = (
        result["assets"] and result["net_income"]
    )

    result["has_debt_candidate"] = (
        result["debt_aggregate"]
        or result["debt_components"]
    )

    result["has_core_variables"] = (
        result["has_profitability"]
        and result["has_debt_candidate"]
    )

    return result


fy2024_results = []
fy2024_errors = []

for row in tqdm(
    financial_pilot.itertuples(index=False),
    total=len(financial_pilot),
    desc="Checking FY2024 coverage"
):
    try:
        result = check_fy2024_coverage(row.cik_str)

        if result is not None:
            result["cik_str"] = row.cik_str
            result["name"] = row.name
            fy2024_results.append(result)

    except requests.RequestException as exc:
        fy2024_errors.append({
            "cik_str": row.cik_str,
            "error": str(exc)
        })

    time.sleep(0.2)


fy2024_coverage = pd.DataFrame(fy2024_results)

variables = [
    "assets",
    "net_income",
    "equity",
    "revenue",
    "debt_aggregate",
    "debt_components",
    "has_profitability",
    "has_debt_candidate",
    "has_core_variables"
]

print("Pilot firms:", len(financial_pilot))
print("Firms with FY2024 asset records:", len(fy2024_coverage))
print("Failed requests:", len(fy2024_errors))

if not fy2024_coverage.empty:

    summary = pd.DataFrame({
        "variable": variables,
        "firms_available": [
            int(fy2024_coverage[v].sum())
            for v in variables
        ]
    })

    summary["pct_of_pilot"] = (
        100 * summary["firms_available"]
        / len(financial_pilot)
    ).round(1)

    display(summary)

Checking FY2024 coverage: 100%|██████████| 30/30 [00:19<00:00,  1.53it/s]

Pilot firms: 30
Firms with FY2024 asset records: 26
Failed requests: 0


,variable,firms_available,pct_of_pilot
0,assets,26,86.7
1,net_income,26,86.7
2,equity,26,86.7
3,revenue,24,80.0
4,debt_aggregate,12,40.0
5,debt_components,15,50.0
6,has_profitability,26,86.7
7,has_debt_candidate,16,53.3
8,has_core_variables,16,53.3


## 13 — Diagnosing Missing Debt Coverage

This section identifies firms with available profitability data
but no recognized debt-related observations.

We also distinguish firms reporting aggregate debt concepts
from firms reporting only component-level debt concepts.

These diagnostics will help determine whether missing debt
coverage reflects alternative reporting conventions, genuinely
debt-free firms, or limitations of our extraction procedure.

In [15]:
# 13 — Diagnose missing debt coverage

diagnostic = fy2024_coverage.copy()

diagnostic["debt_status"] = "No recognized debt tag"

diagnostic.loc[
    diagnostic["debt_aggregate"],
    "debt_status"
] = "Aggregate debt available"

diagnostic.loc[
    ~diagnostic["debt_aggregate"]
    & diagnostic["debt_components"],
    "debt_status"
] = "Components only"

print("Debt reporting categories:")
display(
    diagnostic["debt_status"]
    .value_counts()
    .rename_axis("debt_status")
    .reset_index(name="firms")
)

print("\nFirms with profitability but no recognized debt tag:")

missing_debt = diagnostic.loc[
    diagnostic["has_profitability"]
    & ~diagnostic["has_debt_candidate"],
    ["cik_str", "name", "fiscal_end", "debt_status"]
]

display(missing_debt)

print("\nFirms reporting debt components but no aggregate:")

components_only = diagnostic.loc[
    ~diagnostic["debt_aggregate"]
    & diagnostic["debt_components"],
    ["cik_str", "name", "fiscal_end", "debt_status"]
]

display(components_only)

Debt reporting categories:


,debt_status,firms
0,Aggregate debt available,12
1,No recognized debt tag,10
2,Components only,4



Firms with profitability but no recognized debt tag:


,cik_str,name,fiscal_end,debt_status
0,0001956955,"Unusual Machines, Inc.",2024-12-31,No recognized debt tag
2,0001619856,"Caribou Biosciences, Inc.",2024-12-31,No recognized debt tag
3,0001828962,"Cricut, Inc.",2024-12-31,No recognized debt tag
5,0002022626,"Upstream Bio, Inc.",2024-12-31,No recognized debt tag
6,0001738021,"Compass Therapeutics, Inc.",2024-12-31,No recognized debt tag
11,0001321732,Penumbra Inc,2024-12-31,No recognized debt tag
13,0001781162,Momentus Inc.,2024-12-31,No recognized debt tag
14,0000026058,CTS CORP,2024-12-31,No recognized debt tag
16,0000833640,POWER INTEGRATIONS INC,2024-12-31,No recognized debt tag
17,0001477845,"Annovis Bio, Inc.",2024-12-31,No recognized debt tag



Firms reporting debt components but no aggregate:


,cik_str,name,fiscal_end,debt_status
1,0001687932,"J.Jill, Inc.",2024-02-03,Components only
7,0001323885,"AtriCure, Inc.",2024-12-31,Components only
10,0001600033,"e.l.f. Beauty, Inc.",2024-03-31,Components only
12,0001868941,"Fluence Energy, Inc.",2024-09-30,Components only


## 14 — Investigating Missing Debt Observations

The previous feasibility analysis identified firms with
available profitability information but no recognized
debt-related observations.

This section investigates selected firms individually,
examining their fiscal-year 2024 US-GAAP balance-sheet
concepts.

The objective is to distinguish missing debt disclosures
from alternative reporting conventions and potentially
debt-free firms.

No missing debt values are assigned zero at this stage.

In [16]:
# 14 — Inspect balance-sheet concepts for missing-debt firms

INSPECTION_FIRMS = {
    "Cricut": "0001828962",
    "CTS": "0000026058",
    "Power Integrations": "0000833640"
}

BALANCE_KEYWORDS = (
    "Debt",
    "Borrowing",
    "Loan",
    "Credit",
    "CommercialPaper",
    "NotesPayable",
    "LeaseLiability",
    "Liabilities",
    "Assets"
)


def inspect_balance_sheet_tags(cik, fiscal_year=2024):
    """Inspect US-GAAP balance-sheet facts for a fiscal year."""

    facts = get_company_facts(cik)
    gaap = facts.get("facts", {}).get("us-gaap", {})

    rows = []

    for tag, details in gaap.items():

        if not any(
            keyword.lower() in tag.lower()
            for keyword in BALANCE_KEYWORDS
        ):
            continue

        observations = (
            details.get("units", {}).get("USD", [])
        )

        for obs in observations:

            # Balance-sheet values are point-in-time observations.
            if obs.get("start") is not None:
                continue

            # Focus on original annual filings.
            if (
                obs.get("form") != "10-K"
                or obs.get("fy") != fiscal_year
                or obs.get("fp") != "FY"
                or not obs.get("end", "").startswith(str(fiscal_year))
            ):
                continue

            rows.append({
                "tag": tag,
                "value": obs.get("val"),
                "end": obs.get("end"),
                "filed": obs.get("filed"),
                "accession": obs.get("accn")
            })

    return pd.DataFrame(rows)


inspection_tables = {}

for name, cik in INSPECTION_FIRMS.items():

    df = inspect_balance_sheet_tags(cik)

    inspection_tables[name] = df

    print(f"\n{name}")
    print("-" * 50)

    if df.empty:
        print("No matching observations.")
        continue

    # Retain latest filing for each tag and fiscal-end date.
    df = (
        df.sort_values("filed")
        .drop_duplicates(
            subset=["tag", "end"],
            keep="last"
        )
        .sort_values("tag")
    )

    display(
        df[["tag", "value", "end", "filed"]]
    )


Cricut
--------------------------------------------------


,tag,value,end,filed
0,Assets,693035000,2024-12-31,2025-03-05
1,AssetsCurrent,580214000,2024-12-31,2025-03-05
33,AssetsFairValueDisclosure,336914000,2024-12-31,2025-03-05
34,AvailableForSaleDebtSecuritiesAccumulatedGross...,361000,2024-12-31,2025-03-05
2,DeferredIncomeTaxAssetsNet,39186000,2024-12-31,2025-03-05
16,DeferredIncomeTaxLiabilities,10984000,2024-12-31,2025-03-05
35,DeferredTaxAssetsCapitalLossCarryforwards,102000,2024-12-31,2025-03-05
17,DeferredTaxAssetsDeferredIncome,693000,2024-12-31,2025-03-05
18,DeferredTaxAssetsGoodwillAndIntangibleAssets,164000,2024-12-31,2025-03-05
19,DeferredTaxAssetsGross,50170000,2024-12-31,2025-03-05



CTS
--------------------------------------------------


,tag,value,end,filed
0,AccruedLiabilitiesCurrent,34451000,2024-12-31,2025-02-26
1,Assets,766112000,2024-12-31,2025-02-26
2,AssetsCurrent,244277000,2024-12-31,2025-02-26
3,DeferredIncomeTaxAssetsNet,27591000,2024-12-31,2025-02-26
4,DeferredIncomeTaxLiabilities,33379000,2024-12-31,2025-02-26
...,...,...,...,...
56,OtherAccruedLiabilitiesCurrent,8184000,2024-12-31,2025-02-26
57,OtherAssetsCurrent,18716000,2024-12-31,2025-02-26
58,OtherAssetsNoncurrent,13180000,2024-12-31,2025-02-26
59,OtherLiabilitiesNoncurrent,8662000,2024-12-31,2025-02-26



Power Integrations
--------------------------------------------------


,tag,value,end,filed
0,Assets,828826000,2024-12-31,2025-02-07
1,AssetsCurrent,514039000,2024-12-31,2025-02-07
2,AssetsFairValueDisclosure,252388000,2024-12-31,2025-02-07
3,AvailableForSaleDebtSecuritiesAccumulatedGross...,1023000,2024-12-31,2025-02-07
4,AvailableForSaleDebtSecuritiesAccumulatedGross...,135000,2024-12-31,2025-02-07
5,AvailableForSaleDebtSecuritiesAmortizedCostBasis,248135000,2024-12-31,2025-02-07
52,DebtSecuritiesAvailableForSaleAllowanceForCred...,0,2024-12-31,2025-02-07
6,DebtSecuritiesAvailableForSaleContinuousUnreal...,13244000,2024-12-31,2025-02-07
7,DebtSecuritiesAvailableForSaleContinuousUnreal...,19000,2024-12-31,2025-02-07
8,DebtSecuritiesAvailableForSaleContinuousUnreal...,33777000,2024-12-31,2025-02-07


## 15 — Comparing Leverage Variable Availability

This section compares the availability of two candidate leverage
definitions: interest-bearing debt relative to assets and total
liabilities relative to assets.

Interest-bearing debt is the preferred measure, while total
liabilities provides a broader alternative.

The objective is to evaluate the trade-off between accounting
specificity and sample coverage before constructing the final
analytical dataset.

In [17]:
# 15 — Compare leverage proxy availability

LEVERAGE_TAGS = {
    "assets": ["Assets"],
    "total_liabilities": ["Liabilities"],
    "interest_bearing_debt_candidate": [
        "LongTermDebt",
        "LongTermDebtCurrent",
        "LongTermDebtNoncurrent",
        "DebtCurrent",
        "DebtNoncurrent",
        "ShortTermBorrowings",
        "ShortTermDebt",
        "CommercialPaper",
        "LongTermDebtAndCapitalLeaseObligations",
        "LongTermDebtAndCapitalLeaseObligationsCurrent",
        "LongTermDebtAndCapitalLeaseObligationsNoncurrent"
    ]
}


def check_leverage_coverage(cik, fiscal_end, accession):
    """Check candidate leverage concepts in a selected filing."""

    facts = get_company_facts(cik)
    gaap = facts.get("facts", {}).get("us-gaap", {})

    result = {}

    for variable, tags in LEVERAGE_TAGS.items():
        result[variable] = any(
            any(
                obs.get("accn") == accession
                and obs.get("end") == fiscal_end
                and obs.get("start") is None
                for obs in gaap.get(tag, {})
                    .get("units", {})
                    .get("USD", [])
            )
            for tag in tags
        )

    result["liabilities_leverage_available"] = (
        result["assets"]
        and result["total_liabilities"]
    )

    result["debt_leverage_candidate_available"] = (
        result["assets"]
        and result["interest_bearing_debt_candidate"]
    )

    return result


leverage_results = []
leverage_errors = []

for row in tqdm(
    fy2024_coverage.itertuples(index=False),
    total=len(fy2024_coverage),
    desc="Comparing leverage coverage"
):
    try:
        result = check_leverage_coverage(
            row.cik_str,
            row.fiscal_end,
            row.accession
        )

        result["cik_str"] = row.cik_str
        result["name"] = row.name

        leverage_results.append(result)

    except requests.RequestException as exc:
        leverage_errors.append({
            "cik_str": row.cik_str,
            "error": str(exc)
        })

    time.sleep(0.2)


leverage_coverage = pd.DataFrame(leverage_results)

summary = pd.DataFrame({
    "measure": [
        "Interest-bearing debt candidate",
        "Total liabilities"
    ],
    "firms_available": [
        int(
            leverage_coverage[
                "debt_leverage_candidate_available"
            ].sum()
        ),
        int(
            leverage_coverage[
                "liabilities_leverage_available"
            ].sum()
        )
    ]
})

summary["pct_of_pilot"] = (
    100 * summary["firms_available"]
    / len(financial_pilot)
).round(1)

print("Pilot firms:", len(financial_pilot))
print("Firms evaluated:", len(leverage_coverage))
print("Failed requests:", len(leverage_errors))

display(summary)

Comparing leverage coverage: 100%|██████████| 26/26 [00:14<00:00,  1.83it/s]

Pilot firms: 30
Firms evaluated: 26
Failed requests: 0


,measure,firms_available,pct_of_pilot
0,Interest-bearing debt candidate,16,53.3
1,Total liabilities,24,80.0


## 16 — Extracting Fiscal-Year 2024 Financial Values

This section constructs a preliminary firm-level accounting
dataset using SEC XBRL observations.

The primary leverage measure is total liabilities divided by
total assets. Profitability is measured using return on assets
(ROA), defined as net income divided by total assets.

All observations must correspond to the same fiscal year.
Balance-sheet variables are measured at fiscal year-end,
while income-statement variables cover approximately one year.

The resulting dataset is preliminary and will be validated
before use in statistical modeling.

In [18]:
# 16 — Extract FY2024 accounting values

import numpy as np
import pandas as pd
import time
from tqdm.auto import tqdm

FINANCIAL_VARIABLES = {
    "assets": [
        "Assets"
    ],
    "liabilities": [
        "Liabilities"
    ],
    "equity": [
        "StockholdersEquity",
        "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest"
    ],
    "net_income": [
        "NetIncomeLoss",
        "ProfitLoss"
    ],
    "revenue": [
        "RevenueFromContractWithCustomerExcludingAssessedTax",
        "Revenues",
        "SalesRevenueNet"
    ]
}

BALANCE_VARIABLES = {
    "assets", "liabilities", "equity"
}


def extract_financial_values(cik, fiscal_end, accession):
    """Extract financial values from one selected SEC 10-K."""

    facts = get_company_facts(cik)
    gaap = facts.get("facts", {}).get("us-gaap", {})

    result = {}

    for variable, tags in FINANCIAL_VARIABLES.items():

        result[variable] = np.nan
        result[f"{variable}_tag"] = None

        for tag in tags:

            observations = (
                gaap.get(tag, {})
                .get("units", {})
                .get("USD", [])
            )

            matches = []

            for obs in observations:

                if (
                    obs.get("accn") != accession
                    or obs.get("end") != fiscal_end
                    or obs.get("form") != "10-K"
                ):
                    continue

                if variable in BALANCE_VARIABLES:

                    # Balance-sheet values are point-in-time.
                    if obs.get("start") is not None:
                        continue

                else:

                    # Income-statement values must cover a year.
                    if obs.get("start") is None:
                        continue

                    days = (
                        pd.Timestamp(obs["end"])
                        - pd.Timestamp(obs["start"])
                    ).days

                    if not 330 <= days <= 380:
                        continue

                matches.append(obs)

            if matches:
                result[variable] = matches[0]["val"]
                result[f"{variable}_tag"] = tag
                break

    return result


financial_records = []
financial_errors = []

for row in tqdm(
    fy2024_coverage.itertuples(index=False),
    total=len(fy2024_coverage),
    desc="Extracting financial values"
):
    try:
        values = extract_financial_values(
            row.cik_str,
            row.fiscal_end,
            row.accession
        )

        record = {
            "cik_str": row.cik_str,
            "name": row.name,
            "fiscal_end": row.fiscal_end,
            **values
        }

        financial_records.append(record)

    except requests.RequestException as exc:
        financial_errors.append({
            "cik_str": row.cik_str,
            "error": str(exc)
        })

    time.sleep(0.2)


pilot_financials = pd.DataFrame(financial_records)

# Construct preliminary accounting ratios.
pilot_financials["roa"] = (
    pilot_financials["net_income"]
    / pilot_financials["assets"]
)

pilot_financials["leverage"] = (
    pilot_financials["liabilities"]
    / pilot_financials["assets"]
)

# Prevent division by zero.
pilot_financials.loc[
    pilot_financials["assets"] <= 0,
    ["roa", "leverage"]
] = np.nan

print("Firms extracted:", len(pilot_financials))
print("Failed requests:", len(financial_errors))

print("\nNonmissing observations:")
print(
    pilot_financials[
        ["assets", "liabilities", "net_income",
         "equity", "revenue", "roa", "leverage"]
    ].notna().sum()
)

display(
    pilot_financials[
        [
            "name",
            "fiscal_end",
            "assets",
            "liabilities",
            "net_income",
            "roa",
            "leverage"
        ]
    ]
)

Extracting financial values: 100%|██████████| 26/26 [00:16<00:00,  1.55it/s]

Firms extracted: 26
Failed requests: 0

Nonmissing observations:
assets         26
liabilities    24
net_income     26
equity         26
revenue        24
roa            26
leverage       24
dtype: int64


,name,fiscal_end,assets,liabilities,net_income,roa,leverage
0,"Unusual Machines, Inc.",2024-12-31,16107575,1.289633e+06,-31980468,-1.985430,0.080064
1,"J.Jill, Inc.",2024-02-03,428180000,3.909620e+08,36201000,0.084546,0.913079
2,"Caribou Biosciences, Inc.",2024-12-31,313313000,6.036200e+07,-149105000,-0.475898,0.192657
3,"Cricut, Inc.",2024-12-31,693035000,2.262740e+08,62830000,0.090659,0.326497
4,Leatt Corp,2024-12-31,46488946,NaN,-2204664,-0.047423,NaN
5,"Upstream Bio, Inc.",2024-12-31,481719000,1.186700e+07,-62806000,-0.130379,0.024635
6,"Compass Therapeutics, Inc.",2024-12-31,140403000,1.517000e+07,-49375000,-0.351666,0.108046
7,"AtriCure, Inc.",2024-12-31,609328000,1.483590e+08,-44698000,-0.073356,0.243480
8,"TYSON FOODS, INC.",2024-09-28,37100000000,NaN,800000000,0.021563,NaN
9,ENDRA Life Sciences Inc.,2024-12-31,4452362,1.891996e+06,-11507947,-2.584684,0.424942


## 17 — Validating the Balance-Sheet Identity

This section evaluates whether total liabilities can be
reconstructed from total assets and total equity.

The accounting identity is:

    Assets = Liabilities + Total Equity

We distinguish stockholders' equity from total equity
including noncontrolling interests, where applicable.

The reconstruction will only be used when the accounting
definitions are consistent and the identity is validated.

In [19]:
# 17 — Validate liabilities using the balance-sheet identity

import numpy as np
import pandas as pd
import time
from tqdm.auto import tqdm

EQUITY_VALIDATION_TAGS = [
    "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest",
    "StockholdersEquity",
    "MinorityInterest"
]


def extract_equity_components(cik, fiscal_end, accession):
    """Extract equity components from the selected 10-K."""

    facts = get_company_facts(cik)
    gaap = facts.get("facts", {}).get("us-gaap", {})

    result = {}

    for tag in EQUITY_VALIDATION_TAGS:

        result[tag] = np.nan

        observations = (
            gaap.get(tag, {})
            .get("units", {})
            .get("USD", [])
        )

        matches = [
            obs for obs in observations
            if (
                obs.get("accn") == accession
                and obs.get("end") == fiscal_end
                and obs.get("start") is None
                and obs.get("form") == "10-K"
            )
        ]

        if matches:
            result[tag] = matches[0]["val"]

    return result


equity_records = []

for row in tqdm(
    fy2024_coverage.itertuples(index=False),
    total=len(fy2024_coverage),
    desc="Validating balance-sheet identity"
):
    values = extract_equity_components(
        row.cik_str,
        row.fiscal_end,
        row.accession
    )

    equity_records.append({
        "cik_str": row.cik_str,
        **values
    })

    time.sleep(0.2)


equity_components = pd.DataFrame(equity_records)

balance_validation = pilot_financials.merge(
    equity_components,
    on="cik_str",
    how="left"
)

total_equity_tag = (
    "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest"
)

balance_validation["total_equity"] = (
    balance_validation[total_equity_tag]
)

# If consolidated total equity is unavailable,
# use stockholders' equity plus explicitly reported
# noncontrolling interest, when both are available.
mask = (
    balance_validation["total_equity"].isna()
    & balance_validation["StockholdersEquity"].notna()
    & balance_validation["MinorityInterest"].notna()
)

balance_validation.loc[mask, "total_equity"] = (
    balance_validation.loc[mask, "StockholdersEquity"]
    + balance_validation.loc[mask, "MinorityInterest"]
)

# Where only stockholders' equity is reported,
# retain it as a provisional alternative.
mask_stockholders_only = (
    balance_validation["total_equity"].isna()
    & balance_validation["StockholdersEquity"].notna()
)

balance_validation.loc[
    mask_stockholders_only, "total_equity"
] = balance_validation.loc[
    mask_stockholders_only, "StockholdersEquity"
]

balance_validation["liabilities_implied"] = (
    balance_validation["assets"]
    - balance_validation["total_equity"]
)

balance_validation["identity_difference"] = (
    balance_validation["liabilities"]
    - balance_validation["liabilities_implied"]
)

balance_validation["relative_difference"] = (
    balance_validation["identity_difference"].abs()
    / balance_validation["assets"].replace(0, np.nan)
)

reported = balance_validation.dropna(
    subset=["liabilities", "liabilities_implied"]
)

print("Firms with reported liabilities:", len(reported))

print(
    "Identities consistent within 0.1% of assets:",
    (reported["relative_difference"] < 0.001).sum()
)

print("\nFirms with missing reported liabilities:")

display(
    balance_validation.loc[
        balance_validation["liabilities"].isna(),
        [
            "name",
            "assets",
            "total_equity",
            "liabilities_implied"
        ]
    ]
)

print("\nBalance-sheet identity diagnostics:")

display(
    balance_validation[
        [
            "name",
            "liabilities",
            "liabilities_implied",
            "identity_difference",
            "relative_difference"
        ]
    ]
)

Validating balance-sheet identity: 100%|██████████| 26/26 [00:14<00:00,  1.79it/s]

Firms with reported liabilities: 24
Identities consistent within 0.1% of assets: 23

Firms with missing reported liabilities:


,name,assets,total_equity,liabilities_implied
4,Leatt Corp,46488946,3.826472e+07,8.224229e+06
8,"TYSON FOODS, INC.",37100000000,1.851400e+10,1.858600e+10



Balance-sheet identity diagnostics:


,name,liabilities,liabilities_implied,identity_difference,relative_difference
0,"Unusual Machines, Inc.",1.289633e+06,1.289633e+06,0.0,0.000000
1,"J.Jill, Inc.",3.909620e+08,3.909620e+08,0.0,0.000000
2,"Caribou Biosciences, Inc.",6.036200e+07,6.036200e+07,0.0,0.000000
3,"Cricut, Inc.",2.262740e+08,2.262740e+08,0.0,0.000000
4,Leatt Corp,NaN,8.224229e+06,NaN,NaN
5,"Upstream Bio, Inc.",1.186700e+07,1.186700e+07,0.0,0.000000
6,"Compass Therapeutics, Inc.",1.517000e+07,1.517000e+07,0.0,0.000000
7,"AtriCure, Inc.",1.483590e+08,1.483590e+08,0.0,0.000000
8,"TYSON FOODS, INC.",NaN,1.858600e+10,NaN,NaN
9,ENDRA Life Sciences Inc.,1.891996e+06,1.891996e+06,0.0,0.000000


## 18 — Constructing Validated Leverage Values

This section combines directly reported liabilities with
liabilities reconstructed from the balance-sheet identity.

Reported liabilities are preferred. Reconstruction is accepted
when consolidated total equity is available.

When only stockholders' equity is available, the reconstructed
value remains provisional because noncontrolling interests
may be excluded.

Each observation receives a source flag so that subsequent
analyses can distinguish reported, reconstructed, and
provisional leverage values.

In [20]:
# 18 — Construct leverage values with source tracking

validated_financials = balance_validation.copy()

consolidated_tag = (
    "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest"
)

validated_financials["liabilities_final"] = (
    validated_financials["liabilities"]
)

validated_financials["liabilities_source"] = np.where(
    validated_financials["liabilities"].notna(),
    "reported",
    "missing"
)

# Reconstruct only when consolidated total equity is
# explicitly available, or when stockholders' equity
# and noncontrolling interest are both reported.

has_consolidated_equity = (
    validated_financials[consolidated_tag].notna()
)

has_equity_components = (
    validated_financials["StockholdersEquity"].notna()
    & validated_financials["MinorityInterest"].notna()
)

can_reconstruct = (
    validated_financials["liabilities"].isna()
    & validated_financials["assets"].gt(0)
    & (
        has_consolidated_equity
        | has_equity_components
    )
)

validated_financials.loc[
    can_reconstruct, "liabilities_final"
] = validated_financials.loc[
    can_reconstruct, "liabilities_implied"
]

validated_financials.loc[
    can_reconstruct, "liabilities_source"
] = "reconstructed"

# Identify provisional cases where only
# stockholders' equity is available.

provisional = (
    validated_financials["liabilities"].isna()
    & validated_financials["assets"].gt(0)
    & validated_financials["StockholdersEquity"].notna()
    & ~has_consolidated_equity
    & ~has_equity_components
)

validated_financials.loc[
    provisional, "liabilities_source"
] = "provisional_equity_only"

# Construct final leverage ratio.

validated_financials["leverage_final"] = (
    validated_financials["liabilities_final"]
    / validated_financials["assets"]
)

# Guard against invalid reconstruction.

invalid = (
    validated_financials["liabilities_final"].lt(0)
    | validated_financials["assets"].le(0)
)

validated_financials.loc[
    invalid, "leverage_final"
] = np.nan

print("Liabilities source categories:")

display(
    validated_financials["liabilities_source"]
    .value_counts()
    .rename_axis("source")
    .reset_index(name="firms")
)

print("\nFinal leverage coverage:")

print(
    validated_financials["leverage_final"].notna().sum(),
    "of",
    len(validated_financials),
    "firms with FY2024 assets"
)

print("\nPreviously missing liabilities:")

display(
    validated_financials.loc[
        validated_financials["liabilities"].isna(),
        [
            "name",
            "assets",
            "total_equity",
            "liabilities_implied",
            "liabilities_final",
            "liabilities_source",
            "leverage_final"
        ]
    ]
)

Liabilities source categories:


,source,firms
0,reported,24
1,provisional_equity_only,1
2,reconstructed,1



Final leverage coverage:
25 of 26 firms with FY2024 assets

Previously missing liabilities:


,name,assets,total_equity,liabilities_implied,liabilities_final,liabilities_source,leverage_final
4,Leatt Corp,46488946,3.826472e+07,8.224229e+06,NaN,provisional_equity_only,NaN
8,"TYSON FOODS, INC.",37100000000,1.851400e+10,1.858600e+10,1.858600e+10,reconstructed,0.50097


## 19 — Defining the Eligible Manufacturing Universe

The initial SEC manufacturing universe includes companies
classified under SIC codes 2000–3999.

This section restricts the candidate universe to firms
incorporated in a U.S. state or the District of Columbia
and listed on a recognized U.S. securities exchange.

The restrictions are applied before retrieving detailed
financial statements to reduce unnecessary API requests.

The resulting sample is based on current SEC company
metadata and may therefore be subject to survivorship bias.

In [21]:
# 19 — Apply incorporation and exchange eligibility criteria

US_JURISDICTIONS = {
    "AL", "AK", "AZ", "AR", "CA", "CO", "CT", "DE",
    "FL", "GA", "HI", "ID", "IL", "IN", "IA", "KS",
    "KY", "LA", "ME", "MD", "MA", "MI", "MN", "MS",
    "MO", "MT", "NE", "NV", "NH", "NJ", "NM", "NY",
    "NC", "ND", "OH", "OK", "OR", "PA", "RI", "SC",
    "SD", "TN", "TX", "UT", "VT", "VA", "WA", "WV",
    "WI", "WY", "DC"
}

US_EXCHANGES = {
    "Nasdaq",
    "NYSE",
    "CBOE",
    "TXSE"
}

# Bring exchange information into the manufacturing universe.
# company_universe contains one row per CIK.

manufacturing_eligible = manufacturing_firms.merge(
    company_universe[["cik_str", "exchanges"]],
    on="cik_str",
    how="left",
    validate="one_to_one"
)

manufacturing_eligible["us_incorporated"] = (
    manufacturing_eligible["state_of_incorporation"]
    .isin(US_JURISDICTIONS)
)

manufacturing_eligible["us_exchange"] = (
    manufacturing_eligible["exchanges"]
    .apply(
        lambda exchanges: (
            isinstance(exchanges, list)
            and bool(set(exchanges) & US_EXCHANGES)
        )
    )
)

manufacturing_eligible["eligible"] = (
    manufacturing_eligible["us_incorporated"]
    & manufacturing_eligible["us_exchange"]
)

print("Initial manufacturing candidates:", len(manufacturing_eligible))

print(
    "U.S.-incorporated:",
    manufacturing_eligible["us_incorporated"].sum()
)

print(
    "Listed on selected U.S. exchanges:",
    manufacturing_eligible["us_exchange"].sum()
)

print(
    "Meeting both criteria:",
    manufacturing_eligible["eligible"].sum()
)

print("\nIncorporation jurisdiction distribution:")

display(
    manufacturing_eligible["state_of_incorporation"]
    .fillna("Missing")
    .value_counts()
    .head(20)
    .rename_axis("jurisdiction")
    .reset_index(name="firms")
)

print("\nExamples excluded by incorporation criterion:")

display(
    manufacturing_eligible.loc[
        ~manufacturing_eligible["us_incorporated"],
        [
            "name",
            "state_of_incorporation",
            "exchanges"
        ]
    ].head(15)
)

# Preserve the eligible universe for the next section.

eligible_firms = manufacturing_eligible.loc[
    manufacturing_eligible["eligible"]
].copy()

print("\nFinal eligible CIK count:", len(eligible_firms))

Initial manufacturing candidates: 2407
U.S.-incorporated: 1720
Listed on selected U.S. exchanges: 2073
Meeting both criteria: 1467

Incorporation jurisdiction distribution:


,jurisdiction,firms
0,DE,1210
1,,207
2,NV,173
3,E9,120
4,L3,63
5,A1,49
6,NY,33
7,X0,29
8,FL,28
9,OH,22



Examples excluded by incorporation criterion:


,name,state_of_incorporation,exchanges
6,AKZO NOBEL NV,,[OTC]
39,CANON INC,M0,[OTC]
104,IMPERIAL OIL LTD,,[NYSE]
116,KYOCERA CORP,,[OTC]
135,MSA Safety Inc,,"[NYSE, OTC]"
157,PACKAGING CORP OF AMERICA,,[NYSE]
162,PENTAIR plc,L2,[NYSE]
234,ITT INC.,,[NYSE]
236,UNILEVER PLC,,"[NYSE, OTC]"
244,SUNCOR ENERGY INC,A0,[NYSE]



Final eligible CIK count: 1467


## 20 — Auditing Missing Incorporation Metadata

The initial eligibility screen identified 1,467 manufacturing
firms that are both U.S.-incorporated and listed on selected
U.S. securities exchanges.

However, several recognizable U.S. manufacturers have
missing incorporation metadata.

This section distinguishes confirmed U.S. incorporation,
unknown incorporation, and non-U.S. incorporation.

The objective is to quantify potential sample exclusions
caused by missing metadata before finalizing the universe.

In [22]:
# 20 — Audit missing incorporation metadata

incorporation_audit = manufacturing_eligible.copy()

# Normalize blank or whitespace-only values.
incorporation_audit["incorporation_code"] = (
    incorporation_audit["state_of_incorporation"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.upper()
)

incorporation_audit["incorporation_status"] = np.select(
    [
        incorporation_audit["incorporation_code"].isin(
            US_JURISDICTIONS
        ),
        incorporation_audit["incorporation_code"].eq("")
    ],
    [
        "confirmed_us",
        "unknown"
    ],
    default="non_us"
)

# Restrict the audit to firms on selected exchanges.
exchange_listed = incorporation_audit.loc[
    incorporation_audit["us_exchange"]
].copy()

print("Exchange-listed manufacturing firms:", len(exchange_listed))

print("\nIncorporation status among exchange-listed firms:")

display(
    exchange_listed["incorporation_status"]
    .value_counts()
    .rename_axis("incorporation_status")
    .reset_index(name="firms")
)

print("\nExchange-listed firms with unknown incorporation:")

unknown_incorporation = exchange_listed.loc[
    exchange_listed["incorporation_status"] == "unknown"
].copy()

display(
    unknown_incorporation[
        ["cik_str", "name", "exchanges"]
    ].sort_values("name")
)

print("\nConfirmed U.S. firms:", (
    exchange_listed["incorporation_status"] == "confirmed_us"
).sum())

print("Unknown incorporation:", len(unknown_incorporation))

print("Confirmed non-U.S. incorporation:", (
    exchange_listed["incorporation_status"] == "non_us"
).sum())

Exchange-listed manufacturing firms: 2073

Incorporation status among exchange-listed firms:


,incorporation_status,firms
0,confirmed_us,1467
1,non_us,427
2,unknown,179



Exchange-listed firms with unknown incorporation:


,cik_str,name,exchanges
553,0000926282,"ADTRAN Holdings, Inc.",[Nasdaq]
879,0001272842,AIRGAIN INC,[Nasdaq]
423,0000866291,"ALLEGRO MICROSYSTEMS, INC.",[Nasdaq]
845,0001178670,"ALNYLAM PHARMACEUTICALS, INC.",[Nasdaq]
1014,0001387467,ALPHA & OMEGA SEMICONDUCTOR Ltd,[Nasdaq]
...,...,...,...
1265,0001555280,Zoetis Inc.,[NYSE]
1835,0001819253,biote Corp.,[Nasdaq]
1016,0001388658,"iRhythm Holdings, Inc.",[Nasdaq]
2201,0001970544,iTonic Holdings Ltd,[Nasdaq]



Confirmed U.S. firms: 1467
Unknown incorporation: 179
Confirmed non-U.S. incorporation: 427


## 21 — Investigating Unknown Incorporation Metadata

This section investigates whether missing incorporation
codes can be recovered from alternative fields in SEC
company submissions metadata.

A small set of firms with unknown incorporation status
is inspected before applying any changes to the full
manufacturing universe.

The objective is to reduce unnecessary exclusions while
preserving a reproducible definition of U.S. incorporation.

In [23]:
# 21 — Inspect SEC incorporation metadata for unknown firms

INCORPORATION_TEST_FIRMS = {
    "ADTRAN Holdings": "0000926282",
    "Alnylam Pharmaceuticals": "0001178670",
    "Zoetis": "0001555280",
    "uniQure": "0001590560",
    "Alpha & Omega Semiconductor": "0001387467"
}

incorporation_records = []

for name, cik in INCORPORATION_TEST_FIRMS.items():

    metadata = get_company_metadata(cik)

    incorporation_records.append({
        "name": name,
        "cik_str": cik,
        "state_code": metadata.get(
            "stateOfIncorporation"
        ),
        "state_description": metadata.get(
            "stateOfIncorporationDescription"
        ),
        "business_state": metadata.get(
            "addresses", {}
        ).get("business", {})
        .get("stateOrCountry"),
        "business_country_description": metadata.get(
            "addresses", {}
        ).get("business", {})
        .get("stateOrCountryDescription")
    })

    time.sleep(0.2)

incorporation_test = pd.DataFrame(
    incorporation_records
)

display(incorporation_test)


,name,cik_str,state_code,state_description,business_state,business_country_description
0,ADTRAN Holdings,0000926282,None,None,None,None
1,Alnylam Pharmaceuticals,0001178670,None,None,None,None
2,Zoetis,0001555280,None,None,None,None
3,uniQure,0001590560,None,None,None,None
4,Alpha & Omega Semiconductor,0001387467,None,None,None,None
